# Phase 6 -- The Weld and the Countdown
### *Nine Meters of Silence*, Chapter 1: "Thermal Suicide"

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rjmachauthor/nine-meters-of-silence/blob/main/ch01/notebooks/phase6_countdown.ipynb)

**The claim in the book:** a fresh coat of paint hides an unshielded weld over a ground-down fracture line. The weld caused hydrogen embrittlement, dropping the titanium spar's fracture toughness. Under the transient startup torque spike, Vince runs Paris' Law crack growth cycle by cycle -- the tablet finishes: forty-two seconds.

**On the sourcing:** the Paris' Law coefficient/exponent and the fracture toughness values are real, peer-reviewed, cited in `CALIBRATION.md` with DOIs. The applied stress is a disclosed assumption (no public data exists for a fictional aircraft's operating loads) -- but it's treated correctly as a **bounded transient**, matching the manuscript's own wording, not a stress sustained forever. That distinction matters: sustained indefinitely, even healthy titanium eventually fails at this stress level (just slower). Bounded to the real duration of an actual torque transient, healthy titanium survives and embrittled titanium doesn't -- which is the sharp, correct proof that the weld is what kills this flight.

In [ ]:
import sys, os
if 'google.colab' in sys.modules:
    !git clone -q https://github.com/rjmachauthor/nine-meters-of-silence.git
    sys.path.insert(0, 'nine-meters-of-silence/ch01')
else:
    sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

from physics.crack_growth import (
    simulate_crack_growth, fails_within_transient, stress_intensity_factor,
    C_PARIS, M_PARIS, Y_GEOMETRY,
    K_IC_HEALTHY, K_IC_EMBRITTLED,
    ASSUMED_TRANSIENT_STRESS_MPA, ASSUMED_TRANSIENT_DURATION_S,
)

In [ ]:
ttf_e, a_crit_e, cycles_e = simulate_crack_growth(
    a0_mm=5.8, sigma_mpa=ASSUMED_TRANSIENT_STRESS_MPA, k_ic=K_IC_EMBRITTLED, cycles_per_second=4.0,
)
ttf_h, a_crit_h, cycles_h = simulate_crack_growth(
    a0_mm=5.8, sigma_mpa=ASSUMED_TRANSIENT_STRESS_MPA, k_ic=K_IC_HEALTHY, cycles_per_second=4.0,
)

print(f"Embrittled: fails at {ttf_e:.1f}s (critical length {a_crit_e:.2f}mm)")
print(f"Healthy:    fails at {ttf_h:.1f}s (critical length {a_crit_h:.2f}mm)")
print(f"\nActual torque transient duration: ~{ASSUMED_TRANSIENT_DURATION_S:.0f}s (real estimate for a heavy-lift helicopter's high-power climb-out)")
print()
print(f"Embrittled fails within the transient: {fails_within_transient(5.8, ASSUMED_TRANSIENT_STRESS_MPA, K_IC_EMBRITTLED, 4.0, ASSUMED_TRANSIENT_DURATION_S)}")
print(f"Healthy fails within the transient:    {fails_within_transient(5.8, ASSUMED_TRANSIENT_STRESS_MPA, K_IC_HEALTHY, 4.0, ASSUMED_TRANSIENT_DURATION_S)}")
print("\nThat's the proof: within the actual bounded duration of this transient, only the embrittled mast fails.")

In [ ]:
# Rebuild the crack-length-over-time curves for both scenarios to animate side by side
def crack_history(sigma, k_ic, a0_mm=5.8, max_time=ASSUMED_TRANSIENT_DURATION_S + 20):
    a = a0_mm/1000
    a_crit = (1/np.pi) * (k_ic/(Y_GEOMETRY*sigma))**2
    lengths = [a*1000]
    cycles_list = [0]
    c = 0
    max_c = int(max_time * 4.0) + 10
    while a < a_crit and c < max_c:
        dK = stress_intensity_factor(sigma, a, Y_GEOMETRY)
        da_dN = C_PARIS * (dK ** M_PARIS)
        a += da_dN
        c += 1
        lengths.append(a*1000)
        cycles_list.append(c)
    times = np.array(cycles_list) / 4.0
    return times, np.array(lengths), a_crit*1000

t_e, l_e, ac_e = crack_history(ASSUMED_TRANSIENT_STRESS_MPA, K_IC_EMBRITTLED)
t_h, l_h, ac_h = crack_history(ASSUMED_TRANSIENT_STRESS_MPA, K_IC_HEALTHY)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))
ax.set_xlim(0, ASSUMED_TRANSIENT_DURATION_S + 15)
ax.set_ylim(5.7, max(ac_e, ac_h) * 1.05)
ax.axvline(ASSUMED_TRANSIENT_DURATION_S, color='gray', linestyle=':', label=f'Torque transient ends (~{ASSUMED_TRANSIENT_DURATION_S:.0f}s)')
ax.axhline(ac_e, color='crimson', linestyle='--', alpha=0.5, label=f'Embrittled critical ({ac_e:.1f}mm)')
ax.axhline(ac_h, color='seagreen', linestyle='--', alpha=0.5, label=f'Healthy critical ({ac_h:.1f}mm)')
ax.set_xlabel('Time since transient begins (s)')
ax.set_ylabel('Crack length (mm)')
ax.set_title('Phase 6: Embrittled vs. Healthy Titanium Under the Same Transient')
ax.legend(loc='upper left', fontsize=8)

line_e, = ax.plot([], [], color='crimson', lw=2, label='Embrittled')
line_h, = ax.plot([], [], color='seagreen', lw=2, label='Healthy')
point_e, = ax.plot([], [], 'o', color='crimson', markersize=8)
point_h, = ax.plot([], [], 'o', color='seagreen', markersize=8)

n_frames = 130
max_t = ASSUMED_TRANSIENT_DURATION_S + 15
frame_times = np.linspace(0, max_t, n_frames)

def update(i):
    ft = frame_times[i]
    idx_e = np.searchsorted(t_e, ft)
    idx_h = np.searchsorted(t_h, ft)
    idx_e = min(idx_e, len(t_e)-1)
    idx_h = min(idx_h, len(t_h)-1)
    line_e.set_data(t_e[:idx_e+1], l_e[:idx_e+1])
    line_h.set_data(t_h[:idx_h+1], l_h[:idx_h+1])
    point_e.set_data([t_e[idx_e]], [l_e[idx_e]])
    point_h.set_data([t_h[idx_h]], [l_h[idx_h]])
    return line_e, line_h, point_e, point_h

ani = animation.FuncAnimation(fig, update, frames=n_frames, interval=50, blit=True)
plt.close(fig)
HTML(ani.to_jshtml())

Watch where the vertical gray line (transient ending) falls relative to each curve: the embrittled (red) crack crosses its critical threshold well before the transient ends. The healthy (green) crack is still climbing toward its own, much higher threshold when the transient ends -- it never gets there, because by then the pilot has throttled back to cruise power.

## Try it yourself

In [ ]:
# --- Play with these ---
my_transient_duration_s = 50.0   # book scenario assumption. Try shorter (30s) or longer (70s)
my_fracture_toughness = 46.5     # book value (embrittled): 46.5, healthy would be 73.0
# ------------------------

result = fails_within_transient(5.8, ASSUMED_TRANSIENT_STRESS_MPA, my_fracture_toughness, 4.0, my_transient_duration_s)
print(f"With a {my_transient_duration_s:.0f}s transient and K_IC={my_fracture_toughness}: fails within transient = {result}")